# 59. TabM offered to the stack

**One variable against ledger row 142** (`stack_54_realmlp_raw_fe`, CV 0.968933): the member set.
One candidate, `tabm` from row 143.

Row 143 scored **0.966931**, the fourth best single model here, and correlates with `realmlp10` at
Spearman **0.991615**, which is inside this repo's within-family band of 0.9741 to 0.9981.

## The case against, written first

Row 139 is the direct precedent and it is close to exact: a strong network offered to a stack that
already holds its near-twin returned **+0.000011**. Row 126 offered the best single model in the
repo to a stack holding its near-twin and returned +0.000021. This candidate is weaker than the
incumbent it resembles, by -0.000962, and NOTES.md has refuted mechanism-to-diversity reasoning
twice already.

## The prediction

**+0.00000 to +0.00002, under the floor.** The correlation is the reason, and the gate is being
run anyway because this repo gates on measured blend value rather than on correlation. That rule
exists because row 59 found a model that was null alone and took the second largest weight, and
row 77 found five members that all lost alone and cleared together.

On the rank table, even the optimistic end is under 4 places.


In [1]:
# 37_stack_views.ipynb
# Membership gate for the five vectors produced by notebooks 35 and 36.
# Runs locally: every member vector already lives in artifacts/oof.
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(b for b in [Path.cwd(), *Path.cwd().parents]
            if (b / "data" / "raw" / "train.csv").exists())
O = ROOT / "artifacts" / "oof"
S = ROOT / "submissions"

train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy(np.int8)

# The fold vector is rebuilt rather than loaded, and then checked. A silently different
# fold vector is the one error here that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
FOLD_SHA = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
assert FOLD_SHA == "ec282b0968059676", FOLD_SHA
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")
print(f"fold sha {FOLD_SHA}  VERIFIED")

train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]
fold sha ec282b0968059676  VERIFIED


In [2]:
# Row 94's forty-one minus the duplicate, in row 94's order, then the five candidates.
BASE = [
    ("te42", "te_bag42"), ("te2024", "te_seed2024"), ("te7", "te_seed7"),
    ("te2025", "te_seed2025"), ("te13", "te_seed13"),
    ("anchor", "lgbm_default_anchor_seed42"), ("trees300", "lgbm_trees300_seed42"),
    ("trees1000", "lgbm_trees1000_seed42"), ("trees2000", "lgbm_trees2000_seed42"),
    ("lr010", "lgbm_lr01_n1000_seed42"), ("lr005", "lgbm_lr005_n2000_seed42"),
    ("lr003", "lgbm_lr003_n3333_seed42"),
    ("bag42", "lgbm_bag08_lr005_n2000_seed42"),
    ("bag2024", "lgbm_bag08_lr005_n2000_seed2024"),
    ("bag7", "lgbm_bag08_lr005_n2000_seed7"),
    ("bag2025", "lgbm_bag08_lr005_n2000_seed2025"),
    ("bag13", "lgbm_bag08_lr005_n2000_seed13"),
    ("neural", "neural"), ("cat42", "catboost_te"), ("cat2024", "catboost_te_seed2024"),
    ("cat7", "catboost_te_seed7"), ("cat2025", "catboost_te_seed2025"),
    ("cat13", "catboost_te_seed13"), ("neural_te", "neural_te"),
    ("xgb_te", "xgb_te"), ("xgb2024", "xgb_te_seed2024"), ("xgb7", "xgb_te_seed7"),
    ("xgb2025", "xgb_te_seed2025"), ("xgb13", "xgb_te_seed13"),
    ("pair_top9", "xgb_pair_top9"),
]
BASE += [("cat_nat_c1", "cat_native_c1"), ("cat_nat_c2", "cat_native_c2"),
         ("xgb_raw", "xgb_raw"), ("cat_raw", "cat_raw"),
         ("xgb_te_fe", "xgb_te_fe"), ("cat_te_n4000", "cat_te_n4000"),
         ("xgb_raw_fe", "xgb_raw_fe"), ("cat_raw_n10k", "cat_raw_n10000"),
         ("lgb_raw_fe", "lgb_raw_fe"), ("cat_raw_fe", "cat_raw_fe")]
# lgb_raw is deliberately absent: it is bag42 re-run in another kernel, Pearson 0.999970 on
# logits. See the header and the 2026-08-22 entry in NOTES.md. Dropping it costs -0.000001.
DROPPED = [("lgb_raw", "duplicate of bag42, Pearson 0.999970")]

BASE += [("cat_te_fe", "cat_te_fe"), ("hgb_te_fe", "hgb_te_fe"),
         ("lgb_te_fe", "lgb_te_fe"), ("rf_te_fe", "rf_te_fe"),
         ("neural_lookup", "neural_lookup"), ("et_te_fe", "et_te_fe"),
         ("logit_te_fe", "logit_te_fe")]
BASE += [("xgb_tuned", "xgb_tuned")]
BASE += [("neural_fe", "neural_fe"), ("neural_wide", "neural_wide"),
         ("neural_res", "neural_res")]
BASE += [("realmlp", "realmlp")]
BASE += [("realmlp10", "realmlp10")]
BASE += [("realmlp_raw_fe", "realmlp_raw_fe")]
# THE ONE VARIABLE. Row 142 held these fifty-four.
CAND = [("tabm", "tabm")]


def load(stem, kind):
    # OOF/test vector. Two naming conventions exist in artifacts/oof. The bare
    # "{stem}.npy" form is the OOF side only: the early LightGBM members never had a
    # test .npy written and their test side lives in submissions/. Falling back to the
    # bare name for kind="test" silently returns the OOF vector, which is caught by the
    # length assert below only because train and test differ in length.
    cands = [O / f"{stem}_{kind}.npy"]
    if kind == "oof":
        cands.append(O / f"{stem}.npy")
    for c in cands:
        if c.exists():
            return np.load(c)
    if kind == "test" and (S / f"{stem}.csv").exists():
        df = pd.read_csv(S / f"{stem}.csv")
        # A csv written in a different row order blends perfectly cleanly and is
        # undetectable in the score. Checked rather than assumed.
        assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {stem}"
        return df["addicted_label"].to_numpy()
    raise FileNotFoundError(f"{stem} {kind}")


MEM = BASE + CAND
names = [n for n, _ in MEM]
Poof = {n: load(s, "oof") for n, s in MEM}
Ptest = {n: load(s, "test") for n, s in MEM}

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    assert np.isfinite(Poof[n]).all() and np.isfinite(Ptest[n]).all(), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

# Exact-duplicate quarantine. A duplicated array silently DOUBLES that model's weight.
# Row 59 found xgb_pair_base bit-identical to xgb_te this way and excluded it.
seen = {}
for n in names:
    h = hashlib.md5(np.ascontiguousarray(Poof[n]).tobytes()).hexdigest()
    assert h not in seen, f"{n} is bit-identical to {seen[h]}"
    seen[h] = n
print(f"{len(names)} vectors loaded, no exact duplicates")

# THE CORRELATION SCREEN. Hash equality cannot see one configuration run in two kernels:
# the arrays differ by thread-level numerical noise. bag42 and lgb_raw sat in row 94 at
# Pearson 0.999970 and no check fired. Three lines, and it would have caught them.
Lz = np.column_stack([np.clip(np.log(np.clip(Poof[n], 1e-9, 1 - 1e-9)
                                     / (1 - np.clip(Poof[n], 1e-9, 1 - 1e-9))), -30, 30)
                      for n in names])
Cm = np.corrcoef(((Lz - Lz.mean(0)) / Lz.std(0)).T)
np.fill_diagonal(Cm, 0.0)
near = [(names[i], names[j], Cm[i, j])
        for i in range(len(names)) for j in range(i + 1, len(names))
        if abs(Cm[i, j]) > 0.9999]
print(f"pairs above 0.9999: {len(near)}")
for a, b, r in near:
    print(f"  NEAR-DUPLICATE {a} and {b} at {r:+.6f}")
assert not near, "a near-duplicate pair is present, justify it or drop one"
print(f"removed from row 94's set: {[d[0] for d in DROPPED]}")
hi = sorted(((abs(Cm[i, j]), names[i], names[j])
             for i in range(len(names)) for j in range(i + 1, len(names))),
            reverse=True)[:3]
print("most collinear surviving pairs: "
      + ", ".join(f"{a}/{b} {r:.5f}" for r, a, b in hi))

55 vectors loaded, no exact duplicates


pairs above 0.9999: 0
removed from row 94's set: ['lgb_raw']
most collinear surviving pairs: realmlp/realmlp10 0.99948, cat42/cat7 0.99907, cat2024/cat2025 0.99907


In [3]:
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
IDX = {n: i for i, n in enumerate(names)}
BASE54 = [IDX[n] for n, _ in BASE]

print("candidate solo CV, and disagreement with the members it most resembles:")
print(f"  {'candidate':12} {'solo CV':>10} {'rho vs xgb_te':>15} {'rho vs cat42':>14}")
for n, _ in CAND:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    r1 = pd.Series(Poof[n]).corr(pd.Series(Poof["xgb_te"]), method="spearman")
    r2 = pd.Series(Poof[n]).corr(pd.Series(Poof["cat42"]), method="spearman")
    print(f"  {n:12} {cv:10.6f} {r1:15.6f} {r2:14.6f}")

# For scale: how decorrelated are two members that everyone agrees are near-copies?
r_seed = pd.Series(Poof["xgb_te"]).corr(pd.Series(Poof["xgb2024"]), method="spearman")
print(f"\n  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): {r_seed:.6f}")
print("  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.")
print("  This table is context, not a prediction.")

candidate solo CV, and disagreement with the members it most resembles:
  candidate       solo CV   rho vs xgb_te   rho vs cat42


  tabm           0.966931        0.987202       0.988023



  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): 0.997344
  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.
  This table is context, not a prediction.


In [4]:
def run(cols):
    # Fold-wise logistic combiner. No weight is ever fitted on a row it is scored on.
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    nit = []
    for f in range(5):
        tr, va = folds != f, folds == f
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[np.ix_(tr, cols)], y[tr])
        # A non-converged lbfgs fit reads HIGHER than the truth, so convergence is
        # asserted rather than hoped for. Added 2026-08-21 after the public review
        # flagged it; measured at 38 to 40 iterations, so it has never been close.
        nit.append(int(np.max(clf.n_iter_)))
        oof[va] = clf.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = clf.decision_function(Ltest[:, cols])
        cf[f] = clf.coef_[0]
    assert max(nit) < 2000, f"combiner did not converge, {nit}"
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst, cf, max(nit)


ARMS = {"54_row142": BASE54}
for n, _ in CAND:
    ARMS[f"55_{n}"] = BASE54 + [IDX[n]]

res = {a: run(cols) for a, cols in ARMS.items()}

# The prune, recomputed inside a committed notebook rather than trusted from the audit's
# scratch script. Keeps the top k members of the WINNING arm by absolute coefficient.
_best_cols = ARMS["55_tabm"]
_coef = res["55_tabm"][2].mean(axis=0)
_order = np.argsort(-np.abs(_coef))
for _k in (16, 25, 35):
    if _k < len(_best_cols):
        ARMS[f"prune{_k}"] = sorted(_best_cols[i] for i in _order[:_k])
        res[f"prune{_k}"] = run(ARMS[f"prune{_k}"])
per = {a: r[0] for a, r in res.items()}

# Row 54 compared its base arm against 0.968932, which is row 137's PRUNED value rather
# than the 53-member figure. That labelling slip is recorded in row 139's notes and is
# corrected here: the base arm below IS row 139's fifty-three, and 0.968932 is its CV.
ROW142_CV, ROW140_CV = 0.968933, 0.968944
repro = per["54_row142"].mean() - ROW142_CV
print(f"reproduction of row 142: {per['54_row142'].mean():.6f} vs {ROW142_CV:.6f}"
      f"  delta {repro:+.2e}   {'REPRODUCED' if abs(repro) < 1e-4 else 'FAILED'}")
assert abs(repro) < 1e-4, "base arm does not reproduce row 142, do not log this run"
print(f"combiner max n_iter across all arms: {max(r[3] for r in res.values())} of 2000\n")

hdr = " ".join(f"{'fold ' + str(i):>9}" for i in range(5))
print(f"{'arm':14} {hdr} {'mean':>10} {'sd':>9}")
for a in ARMS:
    print(f"{a:14} " + " ".join(f"{v:9.6f}" for v in per[a])
          + f" {per[a].mean():10.6f} {per[a].std():9.6f}")

reproduction of row 142: 0.968933 vs 0.968933  delta -3.64e-07   REPRODUCED
combiner max n_iter across all arms: 70 of 2000

arm               fold 0    fold 1    fold 2    fold 3    fold 4       mean        sd
54_row142       0.968323  0.969032  0.969040  0.969553  0.968716   0.968933  0.000406
55_tabm         0.968331  0.969038  0.969043  0.969566  0.968724   0.968941  0.000408
prune16         0.968329  0.969028  0.969066  0.969545  0.968708   0.968936  0.000404
prune25         0.968356  0.969049  0.969064  0.969569  0.968734   0.968955  0.000402
prune35         0.968349  0.969046  0.969062  0.969575  0.968734   0.968953  0.000405


In [5]:
FLOOR_MEAN, FLOOR_FOLDS = 0.00005, 4
base_per = per["54_row142"]

print("Paired against row 142's fifty-four. The gate is >= +0.00005 mean AND >= 4/5 folds.\n")
print(f"{'arm':14} {'paired mean':>13} {'paired sd':>11} {'folds':>7} {'t(4)':>8}  gate")
gate = {}
for a in ARMS:
    if a == "54_row142":
        continue
    d = per[a] - base_per
    wins = int((d > 0).sum())
    sd = d.std(ddof=1)
    t = d.mean() / (sd / np.sqrt(5)) if sd > 0 else float("inf")
    fired = bool(d.mean() >= FLOOR_MEAN and wins >= FLOOR_FOLDS)
    gate[a] = fired
    print(f"{a:14} {d.mean():+13.6f} {sd:11.6f} {wins:5d}/5 {t:8.2f}"
          f"  {'FIRES' if fired else 'under floor'}")

Paired against row 142's fifty-four. The gate is >= +0.00005 mean AND >= 4/5 folds.

arm              paired mean   paired sd   folds     t(4)  gate
55_tabm            +0.000008    0.000004     5/5     4.54  under floor
prune16            +0.000003    0.000014     2/5     0.45  under floor
prune25            +0.000022    0.000007     5/5     6.85  under floor
prune35            +0.000021    0.000005     5/5     9.70  under floor


In [6]:
# Coefficients of the best arm, which is where the result actually lives. Row 59's
# lesson: a model that is null on its own can still take a large weight, and where that
# weight comes FROM is the thing worth reading.
best = max((a for a in ARMS if a != "54_row142"), key=lambda a: per[a].mean())
print(f"best arm by CV: {best}   {per[best].mean():.6f}\n")

cols_b, cols_0 = ARMS[best], ARMS["54_row142"]
cb = res[best][2].mean(axis=0)
c0 = res["54_row142"][2].mean(axis=0)
base_map = {names[c]: c0[i] for i, c in enumerate(cols_0)}

rows = []
for i, c in enumerate(cols_b):
    n = names[c]
    was = base_map.get(n, float("nan"))
    rows.append({"member": n, "coef": cb[i], "was": was, "shift": cb[i] - was})
tab = pd.DataFrame(rows).sort_values("coef", ascending=False)
print(tab.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

new = set(n for n, _ in CAND) & set(tab.member)
gained = tab[tab.member.isin(new)]["coef"].sum()
lost = -tab[~tab.member.isin(new)]["shift"].sum()
print(f"\nnew members carry {gained:+.4f} in total")
print(f"the existing fifty-three give up {lost:+.4f} of weight between them")
if abs(gained) > 1e-12:
    print(f"substitution covers {100 * lost / gained:.0f} percent of the new weight")

best arm by CV: prune25   0.968955

       member    coef     was   shift
   cat_nat_c2 +0.2790 +0.2878 -0.0088
    realmlp10 +0.2360 +0.2199 +0.0161
 cat_te_n4000 +0.1959 +0.1395 +0.0564
   xgb_raw_fe +0.1640 +0.1425 +0.0215
    xgb_tuned +0.1422 +0.1341 +0.0081
 cat_raw_n10k +0.1006 +0.0881 +0.0125
         tabm +0.0999     NaN     NaN
    cat_te_fe +0.0937 +0.1092 -0.0154
   lgb_raw_fe +0.0890 +0.0817 +0.0073
neural_lookup +0.0728 +0.0728 -0.0000
        lr003 +0.0726 +0.0414 +0.0312
    hgb_te_fe +0.0534 +0.0650 -0.0115
    pair_top9 +0.0459 +0.0485 -0.0026
    xgb_te_fe +0.0372 +0.0397 -0.0025
      xgb2024 +0.0360 +0.0335 +0.0025
   neural_res +0.0325 +0.0357 -0.0032
      realmlp +0.0092 +0.0333 -0.0241
     et_te_fe -0.0148 -0.0291 +0.0143
       anchor -0.0309 -0.0795 +0.0486
  neural_wide -0.0622 -0.0573 -0.0049
     rf_te_fe -0.0660 -0.0360 -0.0300
     trees300 -0.1276 -0.0908 -0.0368
      cat_raw -0.1340 -0.1575 +0.0235
        cat42 -0.1461 -0.0656 -0.0805
   cat_nat_c1 

In [7]:
# Three decisions, kept separate. Bundling them was the error corrected in row 59.
print("1. GATE")
for a, f in gate.items():
    print(f"     {a:14} {'FIRES' if f else 'under floor'}")

print("\n2. MEMBERSHIP")
print("   A sub-floor addition is still kept and logged as negligible: row 32 kept four")
print("   CatBoost seeds at +0.000014 and row 34 kept neural_te at +0.000043. Membership")
print("   follows the sign and the fold count, not the floor.")
keep = [a for a in ARMS if a != "54_row142"
        and (per[a] - base_per).mean() > 0
        and int(((per[a] - base_per) > 0).sum()) >= 4]
print(f"   arms positive and >= 4/5 folds: {keep if keep else 'none'}")
print(f"   carried forward: {best} at {per[best].mean():.6f}")

print("\n3. SUBMISSION")
SUB = S / "stack_tabm.csv"
# The floor the gate uses, applied to the submission decision too. Row 142 recorded
# that these had different thresholds and that a two-millionth difference wrote a csv.
if per[best].mean() > ROW140_CV + FLOOR_MEAN:
    p = res[best][1].mean(axis=0)
    sub = pd.DataFrame({"id": test["id"].to_numpy(),
                        "addicted_label": (np.argsort(np.argsort(p)) + 0.5) / len(p)})
    assert len(sub) == len(test) and np.isfinite(sub["addicted_label"]).all()
    sub.to_csv(SUB, index=False)
    print(f"   wrote {SUB.name}, {len(sub):,} rows,"
          f" {sub['addicted_label'].nunique():,} distinct")
    print("   AUC reads order only, so the rank transform changes nothing and keeps the")
    print("   file comparable with the earlier stack submissions.")
else:
    print(f"   no submission: best arm {per[best].mean():.6f} does not beat row 140's "
          f"{ROW140_CV:.6f}")

print(f"\nledger lines:\n  name    stack_{best}\n  cv_mean {per[best].mean():.6f}"
      f"\n  cv_std  {per[best].std():.6f}")

1. GATE
     55_tabm        under floor
     prune16        under floor
     prune25        under floor
     prune35        under floor

2. MEMBERSHIP
   A sub-floor addition is still kept and logged as negligible: row 32 kept four
   CatBoost seeds at +0.000014 and row 34 kept neural_te at +0.000043. Membership
   follows the sign and the fold count, not the floor.
   arms positive and >= 4/5 folds: ['55_tabm', 'prune25', 'prune35']
   carried forward: prune25 at 0.968955

3. SUBMISSION
   no submission: best arm 0.968955 does not beat row 140's 0.968944

ledger lines:
  name    stack_prune25
  cv_mean 0.968955
  cv_std  0.000402
